# CSE 251B Blended-Data Pipeline

This notebook prepares blended train shards, trains the nano model with root `val.bin` validation, runs the TA evaluation script, and packages the submission files under `results/<timestamp>` before copying them to Drive.

In [1]:
!nvidia-smi

Wed May 27 10:12:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive And Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import time

DRIVE_ROOT = Path('/content/drive/MyDrive/251B')
DRIVE_DATASET_DIR = DRIVE_ROOT / 'dataset' / 'blend'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'

REPO_DIR = Path('/content/milestone')
LOCAL_DATASET_DIR = REPO_DIR / 'data' / 'blend'

RUN_NAME = 'blend_nano_' + time.strftime('%Y%m%d_%H%M%S')
LOCAL_RESULTS_DIR = REPO_DIR / 'results' / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RESULTS_DIR / RUN_NAME

DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive dataset:', DRIVE_DATASET_DIR)
print('Drive results:', DRIVE_RESULTS_DIR)
print('Repo:', REPO_DIR)
print('Local dataset:', LOCAL_DATASET_DIR)
print('Local run results:', LOCAL_RESULTS_DIR)
print('Drive run results:', DRIVE_RUN_DIR)

Mounted at /content/drive
Drive dataset: /content/drive/MyDrive/251B/dataset/blend
Drive results: /content/drive/MyDrive/251B/results
Repo: /content/milestone
Local dataset: /content/milestone/data/blend
Local run results: /content/milestone/results/blend_nano_20260527_101334
Drive run results: /content/drive/MyDrive/251B/results/blend_nano_20260527_101334


## 2. Prepare Repository

In [3]:
!rm -rf /content/milestone
!git clone -b v3 https://github.com/FufenNan/milestone.git /content/milestone
%cd /content/milestone
!git status --short --branch

LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Cloning into '/content/milestone'...
remote: Enumerating objects: 827, done.
remote: Counting objects: 100% (827/827), done.
remote: Compressing objects: 100% (388/388), done.
remote: Total 827 (delta 461), reused 787 (delta 432), pack-reused 0 (from 0)
Receiving objects: 100% (827/827), 7.79 MiB | 20.51 MiB/s, done.
Resolving deltas: 100% (461/461), done.
/content/milestone
## v3...origin/v3


## 3. Install Dependencies

In [4]:
!pip install -q datasets tiktoken tqdm

In [5]:
!pip install -q "datasets==2.19.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


## 4. Copy Or Prepare Training Blend

In [6]:
DATASET_LOG = LOCAL_RESULTS_DIR / 'dataset_prepare.log'

def log(msg):
    line = f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}'
    print(line)
    with open(DATASET_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

BLEND_SOURCES = {
    'fineweb': {'dir': 'fineweb_edu', 'prefix': 'fineweb', 'min_shards': 20},
    'wikipedia': {'dir': 'wikipedia', 'prefix': 'wikipedia', 'min_shards': 9},
    'arxiv': {'dir': 'papers_arxiv', 'prefix': 'papers_arxiv', 'min_shards': 5},
    'pubmed': {'dir': 'papers_pubmed', 'prefix': 'papers_pubmed', 'min_shards': 2},
    'books': {'dir': 'books', 'prefix': 'books', 'min_shards': 7},
}

def count_shards(path):
    path = Path(path)
    return {
        name: len(sorted((path / spec['dir']).glob(f"{spec['prefix']}_train_*.bin")))
        for name, spec in BLEND_SOURCES.items()
    }

def format_counts(counts):
    return ', '.join(f'{name}={counts.get(name, 0)}' for name in BLEND_SOURCES)

def shard_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob('*_train_*.bin'))

def has_ready_dataset(path):
    counts = count_shards(path)
    return all(counts[name] >= spec['min_shards'] for name, spec in BLEND_SOURCES.items())

def sync_dir(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(['rsync', '-ah', '--info=progress2', f'{src}/', f'{dst}/'], check=True)
    except FileNotFoundError:
        shutil.copytree(src, dst, dirs_exist_ok=True)

SHARD_SIZE = 100_000_000
NUM_PROC = max(1, (os.cpu_count() or 2) // 2)

drive_counts = count_shards(DRIVE_DATASET_DIR)
log(f'Drive shards: {format_counts(drive_counts)}, size={shard_bytes(DRIVE_DATASET_DIR) / 1e9:.2f} GB')

if has_ready_dataset(DRIVE_DATASET_DIR):
    log('Copying blended shards from Drive to repo data/blend...')
    sync_dir(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    log('Drive dataset is incomplete. Preparing blended shards under repo data/blend...')
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    for source_name, source_spec in BLEND_SOURCES.items():
        cmd = [
            'python', '-u', 'data/blend.py',
            '--data-root', str(LOCAL_DATASET_DIR),
            '--sources', source_name,
            '--streaming',
            '--shard-size', str(SHARD_SIZE),
            '--max-shards-per-source', str(source_spec['min_shards']),
            '--num-proc', str(NUM_PROC),
        ]
        log('Running: ' + ' '.join(cmd))
        with open(DATASET_LOG, 'a', encoding='utf-8') as log_file:
            proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end='')
                log_file.write(line)
                log_file.flush()
            ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'Dataset preparation for {source_name} failed with exit code {ret}. See {DATASET_LOG}')
        log(f'Copying prepared {source_name} shards back to Drive...')
        sync_dir(LOCAL_DATASET_DIR / source_spec['dir'], DRIVE_DATASET_DIR / source_spec['dir'])
    log('Copying prepared blended shards back to Drive...')
    sync_dir(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR)

local_counts = count_shards(LOCAL_DATASET_DIR)
log(f'Local shards: {format_counts(local_counts)}, size={shard_bytes(LOCAL_DATASET_DIR) / 1e9:.2f} GB')
assert has_ready_dataset(LOCAL_DATASET_DIR), 'Blended dataset is not ready in repo data/blend.'

[2026-05-27 10:14:26] Drive shards: fineweb=20, wikipedia=9, arxiv=5, pubmed=2, books=7, size=8.60 GB
[2026-05-27 10:14:26] Copying blended shards from Drive to repo data/blend...
[2026-05-27 10:22:09] Local shards: fineweb=20, wikipedia=9, arxiv=5, pubmed=2, books=7, size=8.60 GB


## 5. Train

In [ ]:
TRAIN_STDOUT = LOCAL_RESULTS_DIR / 'train_stdout.log'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

import importlib.util
spec = importlib.util.spec_from_file_location('train_config', REPO_DIR / 'config' / 'config.py')
train_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_config)
print('Optimizer:', getattr(train_config, 'optimizer', 'adamw'))
print('Muon lr:', getattr(train_config, 'muon_lr', None))

cmd = ['python', '-u', 'train.py', '--config', 'config/config.py']
# cmd = ['python', '-u', 'train.py', '--config', 'config/config_10000.py']
print('Run name:', RUN_NAME)
print('Command:', ' '.join(cmd))

with open(TRAIN_STDOUT, 'w', encoding='utf-8') as log_file:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}. See {TRAIN_STDOUT}')

Optimizer: muon
Muon lr: 0.025
Run name: blend_nano_20260527_101334
Command: python -u train.py --config config/config.py
device: cuda
sequence length: 512
gradient accumulation steps: 66
train data mix schedule: fineweb_edu=16, wikipedia=7, papers=5, books=5
train data mix repeats per optimizer step: 2
validation data: /content/milestone/val.bin
parameters: 99,027,320
optimizer: muon | muon tensors: 60 (66,846,720 params) | adamw fallback tensors: 38 (32,180,600 params)
step     0 | val loss 10.9088 | seq_len 512
step     0 | train loss 10.942635 | adamw_lr 7.5000e-07 | muon_lr 6.2500e-05 | norm 7.0339 | seq_len 512 | 45176 tok/s
step    10 | train loss 10.558389 | adamw_lr 8.2500e-06 | muon_lr 6.8750e-04 | norm 6.4286 | seq_len 512 | 163743 tok/s
step    20 | train loss 9.876217 | adamw_lr 1.5750e-05 | muon_lr 1.3125e-03 | norm 4.5187 | seq_len 512 | 163659 tok/s
step    30 | train loss 9.239975 | adamw_lr 2.3250e-05 | muon_lr 1.9375e-03 | norm 3.0467 | seq_len 512 | 163539 tok/s
ste

## 6. Run TA Evaluation

In [ ]:
LOCAL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
EVAL_DATA = REPO_DIR / 'val.bin'
TA_EVAL_JSON = LOCAL_RESULTS_DIR / 'eval_val.json'

assert LOCAL_CHECKPOINT.exists(), f'Missing checkpoint: {LOCAL_CHECKPOINT}'
assert EVAL_DATA.exists(), f'Missing eval data: {EVAL_DATA}'

cmd = [
    'python', '-u', 'evaluate.py',
    '--model_dir', str(REPO_DIR),
    '--checkpoint_filename', 'checkpoints/checkpoint.pt',
    '--data', str(EVAL_DATA),
    '--device', 'cuda',
    '--batch_size', '1',
    '--output_json', str(TA_EVAL_JSON),
]
print('Command:', ' '.join(cmd))

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'TA evaluation failed with exit code {ret}.')

Command: python -u evaluate.py --model_dir /content/milestone --checkpoint_filename checkpoints/checkpoint.pt --data /content/milestone/val.bin --device cuda --batch_size 1 --output_json /content/milestone/results/blend_nano_20260527_014738/eval_val.json

--- Evaluating local model: /content/milestone ---
Loading model from /content/milestone/checkpoints/checkpoint.pt...
Total parameters:       99,027,200
Trainable parameters:   99,027,200
Evaluating on /content/milestone/val.bin (block_size=1024)...

  Perplexity:          25.4062
  Avg loss (nats):     3.234992
  Tokens evaluated:    5,170,176
  Total parameters:    99,027,200
  Eval time:           83.6s

Results written to /content/milestone/results/blend_nano_20260527_014738/eval_val.json


## 7. Package Results And Copy To Drive

In [ ]:
submission_files = {
    REPO_DIR / 'checkpoints' / 'checkpoint.pt': LOCAL_RESULTS_DIR / 'checkpoint.pt',
    REPO_DIR / 'config' / 'config.py': LOCAL_RESULTS_DIR / 'config.py',
    # REPO_DIR / 'config' / 'config_10000.py': LOCAL_RESULTS_DIR / 'config.py',
    REPO_DIR / 'model.py': LOCAL_RESULTS_DIR / 'model.py',
}

optional_logs = [
    REPO_DIR / 'checkpoints' / 'train.log',
    REPO_DIR / 'results' / 'train.log',
    REPO_DIR / 'checkpoints' / 'checkpoint_metadata.pt',
]

for src, dst in submission_files.items():
    assert src.exists(), f'Missing expected file: {src}'
    shutil.copy2(src, dst)

for src in optional_logs:
    if src.exists():
        shutil.copy2(src, LOCAL_RESULTS_DIR / src.name)

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
sync_dir(LOCAL_RESULTS_DIR, DRIVE_RUN_DIR)

print('Local results:', LOCAL_RESULTS_DIR)
print('Drive results:', DRIVE_RUN_DIR)
print('Packaged files:')
for path in sorted(LOCAL_RESULTS_DIR.iterdir()):
    print(' -', path.name)

Local results: /content/milestone/results/blend_nano_20260527_014738
Drive results: /content/drive/MyDrive/251B/results/blend_nano_20260527_014738
Packaged files:
 - checkpoint.pt
 - checkpoint_metadata.pt
 - config.py
 - dataset_prepare.log
 - eval_val.json
 - model.py
 - train.log
 - train_stdout.log


# 8. Exit Colab

In [ ]:
from google.colab import runtime
runtime.unassign()